# 05 — Bipotentiostat (BiStat) & WE32 Multichannel Array

This notebook covers two multi-electrode capabilities:

**BiStat** — a second working electrode (WE2) channel built into some Ivium devices. Both WE1 and WE2 share the same reference and counter electrode, but apply independent potentials and measure independent currents. Common in:
- Rotating Ring-Disk Electrode (RRDE) experiments
- Generator-collector experiments
- Dual-electrode flow cells

**WE32** — a 32-channel working electrode array multiplexer accessory. All 32 channels are measured simultaneously in one `read_we32_currents()` call. Common in:
- Screening arrays (combinatorial electrochemistry)
- Corrosion mapping
- Sensor arrays

### Prerequisites
- Driver open, device connected (see `02_device_and_instance_management.ipynb`)
- **BiStat section**: device must support BiStat (e.g., IviumStat with BiStat option)
- **WE32 section**: WE32 accessory must be attached

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
Pyvium.connect_device()
print("Ready")

---

# Part A — BiStat (Bipotentiostat)

## A1. BiStat Connection Modes

Use connection mode 10 or 11 to enable the BiStat:

| Code | Mode |
|------|------|
| 10 | BiStat 4EL (4-electrode, default for RRDE) |
| 11 | BiStat 2EL (2-electrode) |

WE1 is the disk electrode; WE2 is the ring electrode in a typical RRDE setup.

In [ ]:
Pyvium.set_connection_mode(10)  # BiStat 4EL
print("BiStat 4EL mode selected")

## A2. BiStat Mode and Auto E-Ranging

`set_bistat_mode()` controls two linked settings simultaneously:

| Value | BiStat scanning | WE2 reference frame | Auto E-ranging |
|-------|----------------|---------------------|----------------|
| 0 | Standard (static WE2 offset) | WE2 potential referenced to **RE** | Off |
| 1 | Scanning (WE2 follows WE1 scan) | WE2 potential referenced to **WE1** | On |

- **Mode 0 (standard):** WE2 is held at a fixed offset from the reference electrode. Use for RRDE where the ring needs an absolute potential (e.g., "hold ring at +0.4 V vs. Ag/AgCl").
- **Mode 1 (scanning):** WE2 tracks WE1 with a fixed offset. Use for generator-collector experiments where you want WE2 to stay a fixed number of volts ahead of WE1 throughout a scan.

> `connect_device()` always resets Auto E-ranging to Off (mode 0). If you need scanning mode, call `set_bistat_mode(1)` explicitly after connecting.

In [ ]:
Pyvium.set_bistat_mode(0)  # standard: fixed WE2 potential, auto-Eranging off
print("BiStat mode: standard (static WE2 potential)")

## A3. Setting Current Ranges for WE1 and WE2

WE2 (the ring) typically carries a smaller current fraction, so you can use a finer range.

Range codes for WE2 (BiStat):

| Code | Range |
|------|-------|
| 0 | 10 mA |
| 1 | 1 mA |
| 2 | 100 µA |
| 3 | 10 µA |
| 4 | 1 µA |

In [ ]:
Pyvium.set_current_range(2)      # WE1 (disk): 100 µA
Pyvium.set_we2_current_range(2)  # WE2 (ring): 100 µA
print("WE1 and WE2 current ranges: 100 µA")

## A4. Setting Independent Potentials

- `set_potential(E)` — sets WE1 potential (vs. reference electrode)
- `set_we2_potential(E)` — sets WE2 potential; the reference frame depends on the BiStat mode:
  - **Mode 0** (standard): value is an offset relative to RE
  - **Mode 1** (scanning): value is an offset relative to WE1

In [ ]:
WE1_POTENTIAL = 0.0   # V — disk potential vs. reference
WE2_OFFSET    = 0.4   # V — ring offset relative to WE1

Pyvium.set_potential(WE1_POTENTIAL)
Pyvium.set_we2_potential(WE2_OFFSET)
print(f"WE1 (disk): {WE1_POTENTIAL} V")
print(f"WE2 (ring): WE1 + {WE2_OFFSET} V = {WE1_POTENTIAL + WE2_OFFSET} V vs. ref")

## A5. Dual-Channel Measurement (RRDE Example)

Both channels are read independently. This example simulates an RRDE-style hold: disk is at a reducing potential while the ring is held at an oxidizing potential to detect the product.

In [ ]:
DURATION_S = 10.0
INTERVAL_S = 0.2

Pyvium.set_cell_on()
time.sleep(0.1)

timestamps, I_disk, I_ring = [], [], []
t_start = time.time()

while (time.time() - t_start) < DURATION_S:
    t   = time.time() - t_start
    I1  = Pyvium.get_current()      # WE1 (disk)
    I2  = Pyvium.get_we2_current()  # WE2 (ring)
    timestamps.append(t)
    I_disk.append(I1 * 1e6)   # µA
    I_ring.append(I2 * 1e6)
    time.sleep(INTERVAL_S)

Pyvium.set_cell_off()

# Calculate collection efficiency N = -I_ring / I_disk (ideal RRDE)
avg_disk = np.mean(I_disk)
avg_ring = np.mean(I_ring)
if avg_disk != 0:
    N = -avg_ring / avg_disk
    print(f"Ring collection efficiency N ≈ {N:.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(timestamps, I_disk, 'b-', label=f'WE1 disk (mean {avg_disk:.2f} µA)')
ax.plot(timestamps, I_ring, 'r-', label=f'WE2 ring (mean {avg_ring:.2f} µA)')
ax.set_xlabel("Time (s)")
ax.set_ylabel("Current (µA)")
ax.set_title("BiStat — Dual Channel Measurement")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# Part B — WE32 Multichannel Array

The WE32 accessory connects 32 working electrodes to a single potentiostat. All channels share the same reference and counter electrode.

## B1. Selecting the Active WE32 Channel

`set_we32_channel(index)` selects one channel as the currently active electrode (for methods and direct mode). This does **not** affect `read_we32_currents()`, which always reads all 32 simultaneously.

In [ ]:
Pyvium.set_connection_mode(1)  # back to standard EStat for WE32

# Select channel 1 as the active electrode
Pyvium.set_we32_channel(1)
print("WE32 channel 1 selected as active")

## B2. Setting Offsets

WE32 offsets shift the applied potential on individual channels (–2 V to +2 V). They allow each electrode to be biased slightly differently, useful for:
- Compensating for varying OCV across the array
- Running a potential gradient experiment

**Single channel offset:**

In [ ]:
# Set channel 3 to +0.1 V offset
Pyvium.set_we32_offset(3, 0.1)
print("Channel 3 offset: +0.1 V")

# Use channel_index=0 to apply the same offset to ALL channels
Pyvium.set_we32_offset(0, 0.0)  # reset all to 0 V
print("All channels reset to 0 V offset")

**Bulk offset configuration** — provide a list of values for multiple channels at once:

In [ ]:
# Set offsets for 8 channels using a linear gradient
N_CHANNELS = 8
offsets = [i * 0.05 for i in range(N_CHANNELS)]  # 0 to 0.35 V in 50 mV steps

Pyvium.set_we32_offsets(N_CHANNELS, offsets)
print(f"Applied offsets (V): {[f'{v:.2f}' for v in offsets]}")

## B3. Reading Back Configured Offsets

In [ ]:
readback = Pyvium.get_we32_offsets(N_CHANNELS)
print("Offset readback:")
for i, v in enumerate(readback, start=1):
    print(f"  Channel {i:2d}: {v:+.4f} V")

## B4. Simultaneous 32-Channel Current Readout

`read_we32_currents()` is the core WE32 measurement function. It returns a list of 32 current values, all sampled at the same instant.

> **Simultaneous mode limit:** In WE32 simultaneous mode the maximum sampling rate is **10 samples/s** (minimum 0.1 s interval between calls). Each channel can carry a maximum of ±1 mA. Exceeding these limits will produce unreliable readings. For time-series acquisition, pace your loop accordingly (see B6).

In [ ]:
Pyvium.set_potential(0.2)
Pyvium.set_cell_on()
time.sleep(0.1)  # settle

currents = Pyvium.read_we32_currents()

print(f"WE32 currents (32 channels, all in A):")
for i, I in enumerate(currents, start=1):
    print(f"  Ch {i:2d}: {I:.3e} A")

Pyvium.set_cell_off()

## B5. WE32 Current Map Visualization

A common use case is displaying the 32 currents as a spatial map (e.g., a 4×8 electrode array on a substrate).

In [ ]:
Pyvium.set_potential(0.2)
Pyvium.set_cell_on()
time.sleep(0.1)

currents = Pyvium.read_we32_currents()
Pyvium.set_cell_off()

currents_uA = np.array(currents) * 1e6  # convert to µA

# Reshape as 4 rows × 8 columns
grid = currents_uA.reshape(4, 8)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(grid, cmap='RdYlGn', aspect='auto')
plt.colorbar(im, ax=ax, label='Current (µA)')

# Annotate each cell with channel number
for row in range(4):
    for col in range(8):
        ch = row * 8 + col + 1
        ax.text(col, row, f'{ch}\n{grid[row, col]:.1f}',
                ha='center', va='center', fontsize=8)

ax.set_title("WE32 Current Map (4×8 array)")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.tight_layout()
plt.show()

## B6. Time-Resolved WE32 Acquisition

Call `read_we32_currents()` in a loop to build a time series across all 32 channels.

> Keep the interval ≥ 0.1 s (10 samples/s maximum in simultaneous mode). The loop overhead adds some time on top of `time.sleep`, so set `INTERVAL_S` slightly under your target to account for it.

In [ ]:
DURATION_S = 5.0
INTERVAL_S = 0.5
CHANNELS_TO_SHOW = [0, 7, 15, 31]  # 0-indexed channel indices

Pyvium.set_potential(0.1)
Pyvium.set_cell_on()
time.sleep(0.1)

timeline = []
all_readings = []
t_start = time.time()

while (time.time() - t_start) < DURATION_S:
    t = time.time() - t_start
    snapshot = Pyvium.read_we32_currents()
    timeline.append(t)
    all_readings.append([c * 1e6 for c in snapshot])  # µA
    time.sleep(INTERVAL_S)

Pyvium.set_cell_off()

all_readings = np.array(all_readings)  # shape: (n_timepoints, 32)

fig, ax = plt.subplots(figsize=(9, 4))
for ch_idx in CHANNELS_TO_SHOW:
    ax.plot(timeline, all_readings[:, ch_idx], label=f'Ch {ch_idx+1}')

ax.set_xlabel("Time (s)")
ax.set_ylabel("Current (µA)")
ax.set_title("WE32 — Selected Channels Over Time")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cleanup

In [ ]:
Pyvium.set_cell_off()
Pyvium.disconnect_device()
Pyvium.close_driver()
print("Done")

---

## Summary

### BiStat

| Task | Method |
|------|--------|
| Enable BiStat | `set_connection_mode(10)` or `(11)` |
| Set BiStat mode / auto-Eranging | `set_bistat_mode(0 or 1)` |
| Set WE2 potential offset | `set_we2_potential(V)` |
| Set WE2 current range | `set_we2_current_range(n)` |
| Read WE2 current | `get_we2_current()` → float |

### WE32

| Task | Method |
|------|--------|
| Select active channel | `set_we32_channel(index)` |
| Set single channel offset | `set_we32_offset(index, V)` |
| Set multiple channel offsets | `set_we32_offsets(n, [V, ...])` |
| Read configured offsets | `get_we32_offsets(n)` → list |
| Read all 32 currents | `read_we32_currents()` → list[32] |

## Next

- **`06_method_mode.ipynb`** — Run RRDE, EIS, or any other method file automatically